# D19 S3 A2 - Detection Matching, Precision, Recall, and F1


In [1]:
gt_boxes = [[0,0,10,10], [20,20,30,30], [50,50,60,60]]
pred_boxes = [[0,0,10,10], [1,1,11,11], [20,20,30,30], [100,100,110,110]]
pred_scores = [0.90, 0.85, 0.80, 0.70]


In [2]:
def iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    inter_w = max(0, min(ax2, bx2) - max(ax1, bx1))
    inter_h = max(0, min(ay2, by2) - max(ay1, by1))
    intersection = inter_w * inter_h

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - intersection

    if union == 0:
        return 0.0
    return float(intersection / union)


def match_predictions(pred_boxes, pred_scores, gt_boxes, iou_thresh=0.5):
    order = sorted(range(len(pred_boxes)), key=lambda i: pred_scores[i], reverse=True)
    unmatched_gt = set(range(len(gt_boxes)))
    tp = 0
    fp = 0

    for pred_i in order:
        best_gt = None
        best_iou = -1.0

        for gt_i in unmatched_gt:
            overlap = iou(pred_boxes[pred_i], gt_boxes[gt_i])
            if overlap > best_iou:
                best_iou = overlap
                best_gt = gt_i

        if best_gt is not None and best_iou >= iou_thresh:
            tp += 1
            unmatched_gt.remove(best_gt)
        else:
            fp += 1

    fn = len(unmatched_gt)
    return tp, fp, fn


def precision_recall(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) != 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) != 0 else 0.0
    return float(precision), float(recall)


def f1_score(precision, recall):
    if precision + recall == 0:
        return 0.0
    return float(2 * precision * recall / (precision + recall))


## Verification


In [3]:
tp, fp, fn = match_predictions(pred_boxes, pred_scores, gt_boxes, 0.5)
print(tp, fp, fn)
p, r = precision_recall(tp, fp, fn)
print(round(p, 3), round(r, 3))
print(round(f1_score(p, r), 3))
print(precision_recall(0, 0, 5))


2 2 1
0.5 0.667
0.571
(0.0, 0.0)


## Autograder-style checks


In [4]:
import numpy as np

# 1
assert np.isclose(iou([0,0,10,10], [0,0,10,10]), 1.0)

# 2
assert match_predictions(pred_boxes, pred_scores, gt_boxes, 0.5) == (2, 2, 1)

# 3
p, r = precision_recall(2, 2, 1)
assert np.isclose(p, 0.5) and np.isclose(r, 2/3)

# 4
assert np.isclose(f1_score(0.5, 2/3), 4/7)

# 5
assert precision_recall(0, 0, 5) == (0.0, 0.0)

# 6: perfect detector
perfect_scores = [0.9, 0.8, 0.7]
perfect = match_predictions(gt_boxes, perfect_scores, gt_boxes, 0.5)
assert perfect == (3, 0, 0)
assert precision_recall(*perfect) == (1.0, 1.0)

# 7: box with IoU around 0.68 is a TP at 0.5 and an FP at 0.7
one_gt = [[0,0,10,10]]
one_pred = [[1,1,11,11]]
assert match_predictions(one_pred, [0.9], one_gt, 0.5) == (1, 0, 0)
assert match_predictions(one_pred, [0.9], one_gt, 0.7) == (0, 1, 1)

# 8: one ground-truth box can only be consumed once
dup_preds = [[0,0,10,10], [1,1,11,11]]
assert match_predictions(dup_preds, [0.9, 0.8], one_gt, 0.5) == (1, 1, 0)

print("All 8 autograder-style checks passed.")


All 8 autograder-style checks passed.
